In [2]:
%pip install pandas comtradeapicall openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import comtradeapicall

# 1. pipeline configuration
TARGET_REPORTERS = "702"       # Singapore (Focal Checkpoint)
TARGET_PARTNERS = "356,784"    # India and UAE corridors
YEARS = "2021,2022,2023,2024"  # Multi-year timeline
HS_COMMODITIES = "85,71"       # Electronics & Precious Metals

print("Querying the UN Comtrade Public Gateway with compliant schema...")

try:
    # explicitly pass the strict required positional parameters as None/defaults
    df_raw = comtradeapicall.previewFinalData(
        typeCode="C",           # C = Commodities
        freqCode="A",           # A = Annual data
        clCode="HS",            # Harmonized System
        period=YEARS,
        reporterCode=TARGET_REPORTERS,
        partnerCode=TARGET_PARTNERS,
        partner2Code=None,      # required: Secondary Partner (None skips)
        customsCode=None,       # required: Customs procedure filter (None skips)
        motCode=None,           # required: Mode of Transport (None skips)
        cmdCode=HS_COMMODITIES,
        flowCode="M,X"          # Imports and Exports
    )
    
    if df_raw is not None and not df_raw.empty:
        # create directories and save data locally
        os.makedirs("../data/raw", exist_ok=True)
        output_path = "../data/raw/singapore_corridors_raw.csv"
        df_raw.to_csv(output_path, index=False)
        
        print(f"\n[SUCCESS] Ingested {len(df_raw)} trade ledger records.")
        print(f"[SUCCESS] Data safely cached locally at: {output_path}")
        
        # look at what we successfully captured
        display(df_raw[['period', 'reporterDesc', 'partnerDesc', 'flowDesc', 'primaryValue']].head(5))
    else:
        print("[WARNING] API connected, but returned zero data rows.")

except Exception as e:
    print(f"\n[PIPELINE ERROR] Extraction failed: {str(e)}")

Querying the UN Comtrade Public Gateway with compliant schema...

[SUCCESS] Ingested 16 trade ledger records.
[SUCCESS] Data safely cached locally at: ../data/raw/singapore_corridors_raw.csv


,period,reporterDesc,partnerDesc,flowDesc,primaryValue
0,2021,None,None,None,8.238655e+08
1,2021,None,None,None,9.399888e+08
2,2021,None,None,None,2.822020e+07
3,2021,None,None,None,7.548765e+08
4,2022,None,None,None,1.012759e+09


In [3]:

# STAGE 2: Entity Resolution & Data Cleaning engine
# manually rebuilding text records stripped by the public API gateway


# load the local CSV cache to avoid slamming the UN API endpoints
df_clean = pd.read_csv("../data/raw/singapore_corridors_raw.csv")

# 1. establish strict M49 Mapping Dictionaries for our target lanes
country_map = {702: "Singapore", 356: "India", 784: "United Arab Emirates"}
flow_map = {"M": "Import", "X": "Export"}
cmd_map = {85: "Electronics (HS 85)", 71: "Precious Metals (HS 71)"}

# 2. backfill columns by converting codes to data science string objects
# checking if columns are empty/numeric and mapping them cleanly
if 'reporterCode' in df_clean.columns:
    df_clean['reporterDesc'] = df_clean['reporterCode'].map(country_map)
if 'partnerCode' in df_clean.columns:
    df_clean['partnerDesc'] = df_clean['partnerCode'].map(country_map)
if 'flowCode' in df_clean.columns:
    df_clean['flowDesc'] = df_clean['flowCode'].map(flow_map)
if 'cmdCode' in df_clean.columns:
    df_clean['cmdDesc'] = df_clean['cmdCode'].map(cmd_map)

# 3. clean and organize numerical outputs for presentation
df_clean['primaryValue_USD'] = df_clean['primaryValue'].astype(float)
df_clean['primaryValue_Millions'] = (df_clean['primaryValue_USD'] / 1_000_000).round(2)

# select polished presentation columns
final_columns = [
    'period', 'reporterDesc', 'partnerDesc', 
    'cmdDesc', 'flowDesc', 'primaryValue_Millions'
]

# cache the polished data layer into our folder system
os.makedirs("../data/processed", exist_ok=True)
df_clean[final_columns].to_csv("../data/processed/singapore_outbound_clean.csv", index=False)

print("[SUCCESS] Data Cleaning and Entity Resolution Complete.")
print("Polished data layer stored at: ../data/processed/singapore_outbound_clean.csv\n")

# display the clean dataset rows
display(df_clean[final_columns].head(5))

[SUCCESS] Data Cleaning and Entity Resolution Complete.
Polished data layer stored at: ../data/processed/singapore_outbound_clean.csv



,period,reporterDesc,partnerDesc,cmdDesc,flowDesc,primaryValue_Millions
0,2021,Singapore,United Arab Emirates,Precious Metals (HS 71),Import,823.87
1,2021,Singapore,United Arab Emirates,Precious Metals (HS 71),Export,939.99
2,2021,Singapore,United Arab Emirates,Electronics (HS 85),Import,28.22
3,2021,Singapore,United Arab Emirates,Electronics (HS 85),Export,754.88
4,2022,Singapore,United Arab Emirates,Precious Metals (HS 71),Import,1012.76
